# Molecular Dynamics Histone Deacetylase 8

In this notebook a molecular dynamics approach was used to get a better insight of functional and structural changes of SMC3 caused by mutations. Specifically there is a focus Cornelia de Lange-syndrome, a known pathology.

## Biological Background

Cornelia de Lange Syndrome (CdLS) is a rare genetic disorder, named after the Dutch paediatrician that first described it in 1933 (De Lange, 1933). The disorder affects multiple domains, including the physical, such as development of facial structure and limbs, the cognitive, with the majority displaying an intellectual disability, and the behavioral, such as self-injurious or repetitive behavior (Kline et al., 2018). CdLS is caused by defects in the cohesin protein complex, also known as cohesinopathies (Kline et al., 2018).

Post-translational modification of histones is a largely studied field contributing to a better understanding of transcriptional regulation. One such form of modification is acetylation, which involves the donation of an acetyl group (-COCH₃) from acetyl coenzyme A (Ac-CoA) to a specific site on a protein. There are two distinct forms, acetylation of the N-terminal and acetylation of the ε-amino group of lysine residues (Drazic et al., 2016). Whereas histone acetyltransferases can donate an acetyl group to a target site in a protein, histone deacetylases can remove these groups. Histone acetylation determines transcriptional activity, hyperacetylation is associated with increased activity and hypoacetylation with repression of activity (De Ruijter et al., 2003). (de)acetylation can also take place in other protein structures, HDAC8 was found to deacetylate SMC3. Together with other proteins (SMC1, RAD21 and STAG) SMC3 forms the ring-shaped cohesin complex. This complex plays a vital role in gene regulation, it also ensures sister chromatid cohesion during the cel cycle, to prevent sister chromatids separating before anaphase (Deardorff et al., 2012). Deacetylation of SMC3 during the anaphase is vital as only deacetylated SMC3 can be used to build the cohesin complex, so for efficient reuse of proteins, SMC3 acetylation is essential (Deardorff et al., 2012).


In [3]:
# imports
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align

## Structure HDAC8

HDAC8 is a relatively small enzyme, consisting of 377 residues. It only contains one distinctive domain, which is the histone deacetylase at position 14-324 (colored yellow). The enzyme requires one zinc ion as it is zinc-dependent, the ion can be seen having a central position in the protein. All locations of mutations that will be analyzed are coloured red. Three of them are in the histone deacetylase region and one lies outside. Locations outside of the histone deacetylase domain are colored blue.


### Mutations

The table below contains all mutations that are analyzed in this notebook. All of these mutations are known to be
associated with CdLS5, a subtype of CdLS.

| Mutation | Position |
| -------- | -------- |
| H > R    | 180      |
| T > M    | 311      |
| G > R    | 320      |
| H > R    | 334      |

![](imgs/HDAC8-structure.gif)



## Comparing alphafold variants (and maybe known PDB's)

Using AlphaFold, molecular structure was predicted for 5 different variants of SMC3. Including the wildtype with no
mutations and 4 known mutations that are connected to Cornelia de Lange syndrome. For each variant, 3 predicted
structures are compared, to compare and see if AlphaFold predictions for molecular structure of (mutated) SMC3 are
consistent.



### G320R
Loading all AlphaFold predictions and aligning them to see if AlphaFold predictions are consistent:
```
load G320R-0.pdb
load G320R-1.pdb
load G320R-2.pdb

alignto G320R-0
```
Below the result can be seen, the different predictions are clearly very closely related, with only minor differences
 in structure. The zinc ions seems to be completely overlapping for all three predictions. In the image, prediction 1
  is green,
 prediction 2 is
 blue and
 prediction 3 is pink.
The RMSD of G320R-0/G320R-1 = 0.472 and the RMSD of G320R-0/G320R-2 = 0.392

![](imgs/G320R-align.png)


## Run settings for simulation

As the AlphaFold predictions seem to be consistent, further analyzing can be done by simulating molecular dynamics. This is done by computing 250μs of protein movement using GROMACS (Bekker et al., 1993). This software simulates protein movement in a simulated repetitive space surrounding the protein, filled with water. Various forcefields can be used to calculate molecular dynamics, in this notebook CHARMM (Brooks et al., 2009) was chosen, as it is more suited for calculations involving ions, as opposed to the AMBER forcefields (Case et al., 2025), where it is necessary to immobilize the ion.



In order to visualize the molecular movement of HDAC8 and its mutated forms, first all hydrogen molecules and ions must be removed, with the exception of the cofactor Zn2+ ion. To do this, an index file needs to be created, for the second WT prediction this would be:

`gmx make_ndx -f WT-1.gro -o index.ndx`

This gives the following selection:

```
There are:   377    Protein residues
There are:     1        Ion residues
Analysing Protein...

  0 System              :  5806 atoms
  1 Protein             :  5805 atoms
  2 Protein-H           :  2936 atoms
  3 C-alpha             :   377 atoms
  4 Backbone            :  1131 atoms
  5 MainChain           :  1509 atoms
  6 MainChain+Cb        :  1853 atoms
  7 MainChain+H         :  1869 atoms
  8 SideChain           :  3936 atoms
  9 SideChain-H         :  1427 atoms
 10 Prot-Masses         :  5805 atoms
 11 non-Protein         :     1 atoms
 12 Ion                 :     1 atoms

 nr : group      '!': not  'name' nr name   'splitch' nr    Enter: list groups
 'a': atom       '&': and  'del' nr         'splitres' nr   'l': list residues
 't': atom type  '|': or   'keep' nr        'splitat' nr    'h': help
 'r': residue              'res' nr         'chain' char
 "name": group             'case': case sensitive           'q': save and quit
 'ri': residue index
```

Select 1 | 12 and an index file containing the main system is created. Now when making a reduces trajectory file and pdb file this index can be used.


For each output folder of GROMACS the command below can be run, as only one MD.tpr and MD.xtc file exist per folder. This wil generate a trajectory file containing only the simulated movement of the atoms of the protein and the Zn2+ ion.

`gmx trjconv -s *MD.tpr -f *MD.xtc -pbc mol -o protein_.xtc -n index.ndx`

Select the final group in the selection, this is the group that was created in the index file (see group 13).

```
Select group for output
Group     0 (         System) has  5806 elements
Group     1 (        Protein) has  5805 elements
Group     2 (      Protein-H) has  2936 elements
Group     3 (        C-alpha) has   377 elements
Group     4 (       Backbone) has  1131 elements
Group     5 (      MainChain) has  1509 elements
Group     6 (   MainChain+Cb) has  1853 elements
Group     7 (    MainChain+H) has  1869 elements
Group     8 (      SideChain) has  3936 elements
Group     9 (    SideChain-H) has  1427 elements
Group    10 (    Prot-Masses) has  5805 elements
Group    11 (    non-Protein) has     1 elements
Group    12 (            Ion) has     1 elements
Group    13 (    Protein_Ion) has  5806 elements
```

Besides an adjusted trajectory file, an adjusted pdb is also needed. This can be done by 'dumping' the first frame of the trajectory with:

`gmx trjconv -s *MD.tpr -f *MD.xtc -pbc mol -o protein-ion.pdb -dump 0 -n index.ndx`

Again select the same group as selected for the trajectory file.


One more time a new trajectory needs to be generated, as the molecules of the protein and the ion are seen as two different groups, meaning that with the periodic boundary conditions set to `mol`, the protein/ion will only be set back to center when they reach the boundary. The Zn ion sits centrally in the protein, so the protein will always hit the border first and be set back to center before the Zn ion reaches the boundary. In order to fix this, the 'jumping' or resetting of the protein to center has to be disabled with `-pbc nojump`. The downside is that it will no longer be possible to visualize the systems, as the protein will keep moving away, but after doing this, the analysis will be able to be correctly performed.

`gmx trjconv -s *MD.tpr -f *MD.xtc -pbc nojump -o protein_.xtc -n index.ndx`

Now using the adjusted pdb in combination with the adjusted xtc (with non-jumping systems), the movement of protein can be simulated over time (250ns).

Calculating RMSF

In [ ]:
def calculate_RMSF(pdb, xtc):
    # Maak 'average structure' (reference), door eerste frame te alignen en gemiddelde te nemen van coordinaten
    u = mda.Universe(pdb, xtc)

    average = align.AverageStructure(u, u, select='protein and name CA',
                                     ref_frame=0).run()
    ref = average.results.universe

    # Bereken RMSF
    c_alphas = u.select_atoms('protein and name CA')
    R = rms.RMSF(c_alphas).run()

    return c_alphas, R, u


c_alphas_wildtype, R_wildtype,_ = calculate_RMSF("../WT-1/protein_ion.pdb", "../WT-1/protein_ion.xtc")
c_alphas_G320R, R_G320R,_ = calculate_RMSF("../G320R-1/protein_ion.pdb","../G320R-1/protein_ion.xtc")
c_alphas_H180R, R_H180R,_ = calculate_RMSF("../H180R-1/protein_ion.pdb", "../H180R-1/protein_ion.xtc")
c_alphas_H334R, R_H334R,_ = calculate_RMSF("../H334R-1/protein_ion.pdb", "../H334R-1/protein_ion.xtc")
c_alphas_T311M, R_T311M,_ = calculate_RMSF("../T311M-1/protein_ion.pdb", "../T311M-1/protein_ion.xtc")



# Sources
- Drazic, A., Myklebust, L. M., Ree, R., & Arnesen, T. (2016). The world of protein acetylation. Biochimica Et Biophysica Acta (BBA) - Proteins And Proteomics, 1864(10), 1372–1401. https://doi.org/10.1016/j.bbapap.2016.06.007
- De Ruijter, A. J., Van Gennip, A. H., Caron, H. N., Kemp, S., & Van Kuilenburg, A. B. (2003). Histone deacetylases (HDACs): characterization of the classical HDAC family. Biochemical Journal, 370(3), 737–749. https://doi.org/10.1042/bj20021321
- Deardorff, M. A., Bando, M., Nakato, R., Watrin, E., Itoh, T., Minamino, M., Saitoh, K., Komata, M., Katou, Y., Clark, D., Cole, K. E., De Baere, E., Decroos, C., Di Donato, N., Ernst, S., Francey, L. J., Gyftodimou, Y., Hirashima, K., Hullings, M., . . . Shirahige, K. (2012). HDAC8 mutations in Cornelia de Lange syndrome affect the cohesin acetylation cycle. Nature, 489(7415), 313–317. https://doi.org/10.1038/nature11316
- Bekker, Henk & Berendsen, Herman & Dijkstra, E.J. & Achterop, S. & Drunen, Rudi & van der Spoel, David & Sijbers, A. & Keegstra, H. & Reitsma, B. & Renardus, M.K.R.. (1993). Gromacs: A parallel computer for molecular dynamics simulations. Physics Computing. 92. 252-256.
- Brooks, B. R., Brooks, C. L., Mackerell, A. D., Nilsson, L., Petrella, R. J., Roux, B., Won, Y., Archontis, G., Bartels, C., Boresch, S., Caflisch, A., Caves, L., Cui, Q., Dinner, A. R., Feig, M., Fischer, S., Gao, J., Hodoscek, M., Im, W., . . . Karplus, M. (2009). CHARMM: The biomolecular simulation program. Journal Of Computational Chemistry, 30(10), 1545–1614. https://doi.org/10.1002/jcc.21287
- D.A. Case, H.M. Aktulga, K. Belfon, I.Y. Ben-Shalom, J.T. Berryman, S.R. Brozell, F.S. Carvahol, D.S. Cerutti, T.E. Cheatham, III, G.A. Cisneros, V.W.D. Cruzeiro, T.A. Darden, N. Forouzesh, M. Ghazimirsaeed, G. Giambaşu, T. Giese, M.K. Gilson, H. Gohlke, A.W. Goetz, J. Harris, Z. Huang, S. Izadi, S.A. Izmailov, K. Kasavajhala, M.C. Kaymak, I. Kolossv\'a ry, A. Kovalenko, T. Kurtzman, T.S. Lee, P. Li, Z. Li, C. Lin, J. Liu, T. Luchko, R. Luo, M. Machado, M. Manathunga, K.M. Merz, Y. Miao, O. Mikhailovskii, G. Monard, H. Nguyen, K.A. O'Hearn, A. Onufriev, F. Pan, S. Pantano, A. Rahnamoun, D.R. Roe, A. Roitberg, C. Sagui, S. Schott-Verdugo, A. Shajan, J. Shen, C.L. Simmerling, N.R. Skrynnikov, J. Smith, J. Swails, R.C. Walker, J. Wang, J. Wang, X. Wu, Y. Wu, Y. Xiong, Y. Xue, D.M. York, C. Zhao, Q. Zhu, and P.A. Kollman (2025), Amber 2025, University of California, San Francisco.
- De Lange, C. (1933). Surun type nouveau degeneration (typus Amestelodamensis). Arch Med Enfants, 36, 713-719.
- Kline, A. D., Moss, J. F., Selicorni, A., Bisgaard, A., Deardorff, M. A., Gillett, P. M., Ishman, S. L., Kerr, L. M., Levin, A. V., Mulder, P. A., Ramos, F. J., Wierzba, J., Ajmone, P. F., Axtell, D., Blagowidow, N., Cereda, A., Costantino, A., Cormier-Daire, V., FitzPatrick, D., . . . Hennekam, R. C. (2018). Diagnosis and management of Cornelia de Lange syndrome: first international consensus statement. Nature Reviews Genetics, 19(10), 649–666. https://doi.org/10.1038/s41576-018-0031-0